# Customer Churn Prediction and Retention Analytics
**Category 03: Machine Learning | Project 02**

This walkthrough provides a comprehensive guide to building a production-grade churn prediction pipeline for subscription businesses.

### Key Highlights:
- **Data Cleaning & Deduplication**: Handling blank strings, missing values, and corrupted records.
- **Domain Feature Engineering**: Creating tenure ratios, service count indicators, and tech support bundling features.
- **Class Imbalance Handling**: Benchmarking algorithms with `class_weight='balanced'` and stratified splitting.
- **Algorithms Trained**: Logistic Regression, Decision Tree Classifier, and Random Forest Classifier.
- **Comprehensive Metrics**: Accuracy, Precision, Recall, F1-Score, ROC-AUC, and PR-AUC.
- **Business Economics**: Evaluating the asymmetric cost of False Positives vs. False Negatives.
- **Inference & Scoring Engine**: Production risk scoring with customizable decision thresholds.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_churn_data
from src.data_preprocessing import clean_churn_data, prepare_churn_train_test_data
from src.model_training import train_churn_classifiers, save_best_churn_model
from src.model_evaluation import evaluate_churn_models, compare_churn_models, plot_churn_evaluations
from src.predictor import ChurnPredictor

print("Environment initialized successfully.")

## 1. Data Ingestion
We load the subscriber dataset containing demographics, account settings, and service usage.

In [ ]:
raw_df = load_churn_data()
print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head()

## 2. Data Cleaning and Feature Engineering
We resolve duplicates, impute missing values, and engineer domain features.

In [ ]:
clean_df, report = clean_churn_data(raw_df)
print(f"Cleaning Report: {report}")
clean_df[["tenure_months", "tenure_years", "monthly_charges", "total_charges", "services_count", "churn"]].describe()

## 3. Stratified Train-Test Splitting
Using class stratification prevents target drift between training and testing sets.

In [ ]:
X_train, X_test, y_train, y_test = prepare_churn_train_test_data(clean_df, test_size=0.2, random_state=42)
print(f"Train samples: {len(X_train)} (Churn rate: {y_train.mean():.2%})")
print(f"Test samples:  {len(X_test)} (Churn rate: {y_test.mean():.2%})")

## 4. Model Training with Class Balancing
We train Logistic Regression, Decision Tree, and Random Forest using Scikit-learn pipelines.

In [ ]:
models = train_churn_classifiers(X_train, y_train, random_state=42)
for name in models:
    print(f"[+] Trained: {name}")

## 5. Model Evaluation and Benchmarking
Benchmarking candidate models across Accuracy, Precision, Recall, F1, ROC-AUC, and PR-AUC.

In [ ]:
summary_df, predictions, probabilities = evaluate_churn_models(models, X_test, y_test)
summary_df

## 6. Business Impact: False Positives vs False Negatives
- **Type I Error (False Positive)**: Loyal customer flagged as churner. Proactive incentive or outreach given. Low financial penalty ($10-$25 discount).
- **Type II Error (False Negative)**: Churning customer missed by model. Abrupt cancellation results in lost Customer Lifetime Value (CLV: $800-$1,500+).
Because False Negatives are substantially more expensive, subscription businesses tune thresholds to maximize Recall.

## 7. Model Persistence and Real-Time Scoring
Serialize the best pipeline and execute single-customer inference.

In [ ]:
best_name = compare_churn_models(summary_df)
best_pipe = models[best_name]
best_metrics = summary_df[summary_df["Model"] == best_name].iloc[0].to_dict()
saved_path = save_best_churn_model(best_pipe, best_name, best_metrics)
print(f"Serialized best model to: {saved_path}")

predictor = ChurnPredictor(model_path=saved_path)
sample_cust = {
    "gender": "Female",
    "senior_citizen": "No",
    "partner": "No",
    "dependents": "No",
    "tenure_months": 2,
    "phone_service": "Yes",
    "multiple_lines": "No",
    "internet_service": "Fiber optic",
    "online_security": "No",
    "online_backup": "No",
    "device_protection": "No",
    "tech_support": "No",
    "streaming_tv": "Yes",
    "streaming_movies": "Yes",
    "contract": "Month-to-month",
    "paperless_billing": "Yes",
    "payment_method": "Electronic check",
    "monthly_charges": 89.50,
    "total_charges": 179.00,
}
result = predictor.score_customer(sample_cust, threshold=0.45)
for k, v in result.items():
    print(f"{k}: {v}")